# Phase 4 — Systematic Evaluation

Quantitative, thesis-ready evaluation of the four Tier-A contributions. Produces:

1. **Conformal coverage calibration curve** — empirical vs target across `alpha in {0.05, 0.10, 0.15, 0.20, 0.25, 0.30}`, marginal + per-battery.
2. **Ablation table** — MSE-only baseline / +PINN loss / numeric-only / full multimodal, on the cross-battery split.
3. **Weibull PIT histogram** for RUL calibration (flat histogram = well-calibrated survival model).
4. **RAP diagnosis cost-benefit** — does swapping template descriptions for retrieval-augmented text move downstream SOH regression error?

Artefacts written to `results/phase4_evaluation/`.

## 0. Setup

In [ ]:
import importlib.util, os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

def _load_setup():
    cwd = Path.cwd().resolve()
    candidates = [cwd / 'notebooks' / 'notebook_setup.py', cwd / 'notebook_setup.py']
    if cwd.name == 'notebooks':
        candidates.insert(0, cwd / 'notebook_setup.py')
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location('_notebook_setup', fp)
            mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError('notebooks/notebook_setup.py missing')

PROJECT_ROOT = _load_setup().setup()

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
np.random.seed(0); torch.manual_seed(0)
OUT = Path('results/phase4_evaluation'); OUT.mkdir(parents=True, exist_ok=True)
print('outputs ->', OUT)

In [ ]:
# Reuse the setup from the phase4_tier_a_novelty notebook: features + embeddings cache + saved model.
features_path = 'data/phase3/combined_nasa_mendeley.csv' if os.path.exists('data/phase3/combined_nasa_mendeley.csv') else 'data/phase2/harmonized/all_features.csv'
df = pd.read_csv(features_path).reset_index(drop=True)
exclude_cols = {'battery_id','cycle_number','original_index','soh','has_spikes','dataset','test_condition','round_number'}
feature_cols = [c for c in df.columns if c not in exclude_cols]

splits = {'train': [], 'cal': [], 'val': [], 'test': []}
for bat_id, group in df.groupby('battery_id'):
    g = group.sort_values('cycle_number').reset_index(drop=True)
    n = len(g)
    i_train = int(n*0.60); i_cal = int(n*0.75); i_val = int(n*0.85)
    splits['train'].append(g.iloc[:i_train])
    splits['cal'].append(g.iloc[i_train:i_cal])
    splits['val'].append(g.iloc[i_cal:i_val])
    splits['test'].append(g.iloc[i_val:])
for k in splits: splits[k] = pd.concat(splits[k], ignore_index=True)
print({k: len(v) for k,v in splits.items()})

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(np.nan_to_num(splits['train'][feature_cols].values))
def to_seq(arr, seq_len=50):
    arr = np.nan_to_num(arr); arr = scaler.transform(arr)
    seq = np.tile(arr[:, None, :], (1, seq_len, 1))
    seq *= np.linspace(1.0, 0.95, seq_len)[None, :, None]
    return seq.astype(np.float32)
X_seq = {k: to_seq(splits[k][feature_cols].values) for k in splits}
y     = {k: splits[k]['soh'].values.astype(np.float32) for k in splits}

EMB_CACHE = Path('data/phase4/embeddings_cache.npz')
assert EMB_CACHE.exists(), 'Run notebooks/phase4_tier_a_novelty.ipynb first to populate the embedding cache.'
blob = np.load(EMB_CACHE, allow_pickle=True)
emb = {k: blob[k].astype(np.float32) for k in ('train','cal','val','test')}
train_texts = list(blob['train_texts'])
{k: v.shape for k,v in emb.items()}

In [ ]:
# Load the base cross-attention model; wrap in a lightweight predict function.
from cross_attention_model import CrossAttentionSOHModel
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('models/cross_attention_model.pt', map_location=device, weights_only=False)
cfg = ckpt['config']
base_model = CrossAttentionSOHModel(
    numeric_input_dim=cfg['numeric_input_dim'], semantic_input_dim=cfg['semantic_input_dim'],
    hidden_dim=cfg['hidden_dim'], n_attention_heads=cfg['n_attention_heads'],
    n_transformer_layers=cfg['n_transformer_layers']).to(device)
base_model.load_state_dict(ckpt['model_state_dict']); base_model.eval()

@torch.no_grad()
def predict_soh_batch(seq_np, emb_np, batch=128):
    preds = []
    for i in range(0, len(seq_np), batch):
        s = torch.from_numpy(seq_np[i:i+batch]).to(device)
        e = torch.from_numpy(emb_np[i:i+batch]).to(device)
        preds.append(base_model(s, e)['soh'].cpu().numpy())
    return np.concatenate(preds)

preds = {k: predict_soh_batch(X_seq[k], emb[k]) for k in splits}
for k in ('train','cal','val','test'):
    e = preds[k] - y[k]
    print(f'{k:5s}  MAE={np.mean(np.abs(e)):6.3f}  RMSE={np.sqrt(np.mean(e**2)):6.3f}')

## 1. Conformal coverage calibration curve

For each target coverage 1-alpha, calibrate on the `cal` split, measure empirical coverage on the `test` split. A well-calibrated method sits on the y=x line within finite-sample noise.

In [ ]:
from phase4.conformal import (SplitConformalRegressor, MondrianConformalRegressor,
                               empirical_coverage, grouped_empirical_coverage)

def predict_pair(pair):
    seq_np, emb_np = pair
    return predict_soh_batch(seq_np, emb_np)

alphas = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
rows = []
per_battery_rows = []
for a in alphas:
    marg = SplitConformalRegressor(predict_pair, alpha=a).calibrate((X_seq['cal'], emb['cal']), y['cal'])
    iv_m = marg.predict_interval((X_seq['test'], emb['test']))
    m_stats = empirical_coverage(iv_m, y['test'])

    mond = MondrianConformalRegressor(predict_pair, alpha=a).calibrate(
        (X_seq['cal'], emb['cal']), y['cal'], splits['cal']['battery_id'].values)
    iv_mo = mond.predict_interval((X_seq['test'], emb['test']), splits['test']['battery_id'].values)
    mo_stats = empirical_coverage(iv_mo, y['test'])
    grouped = grouped_empirical_coverage(iv_mo, y['test'], splits['test']['battery_id'].values)

    rows.append({
        'alpha': a, 'target_coverage': 1-a,
        'marginal_coverage': m_stats['empirical_coverage'], 'marginal_width': m_stats['mean_width'],
        'mondrian_coverage': mo_stats['empirical_coverage'], 'mondrian_width': mo_stats['mean_width'],
    })
    for bat, s in grouped.items():
        per_battery_rows.append({'alpha': a, 'target_coverage': 1-a, 'battery_id': bat,
                                 'empirical_coverage': s['empirical_coverage'], 'mean_width': s['mean_width'], 'n': s['n']})

cov_df = pd.DataFrame(rows)
per_bat_df = pd.DataFrame(per_battery_rows)
cov_df.to_csv(OUT / 'conformal_coverage.csv', index=False)
per_bat_df.to_csv(OUT / 'conformal_coverage_per_battery.csv', index=False)
cov_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='ideal (y=x)')
ax.plot(cov_df['target_coverage'], cov_df['marginal_coverage'], 'o-', label='marginal')
ax.plot(cov_df['target_coverage'], cov_df['mondrian_coverage'], 's-', label='Mondrian (per-battery)')
ax.set_xlabel('Target coverage 1 - alpha'); ax.set_ylabel('Empirical coverage (test)')
ax.set_xlim(0.6, 1.0); ax.set_ylim(0.6, 1.02); ax.grid(alpha=0.3); ax.legend()
ax.set_title('Conformal calibration curve')

ax = axes[1]
ax.plot(cov_df['target_coverage'], cov_df['marginal_width'], 'o-', label='marginal')
ax.plot(cov_df['target_coverage'], cov_df['mondrian_width'], 's-', label='Mondrian')
ax.set_xlabel('Target coverage 1 - alpha'); ax.set_ylabel('Mean interval width (SOH %)')
ax.grid(alpha=0.3); ax.legend(); ax.set_title('Interval width vs coverage')
plt.tight_layout(); plt.savefig(OUT / 'conformal_calibration.png', dpi=150); plt.show()

In [ ]:
# Per-battery coverage heatmap: reveals where the model's uncertainty is mis-calibrated.
pivot = per_bat_df.pivot(index='battery_id', columns='target_coverage', values='empirical_coverage')
fig, ax = plt.subplots(figsize=(7, 0.4*len(pivot)+1.5))
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0.4, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels([f'{c:.2f}' for c in pivot.columns])
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8, color='black')
ax.set_xlabel('Target coverage'); ax.set_ylabel('Battery')
ax.set_title('Per-battery empirical coverage (Mondrian calibration)')
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout(); plt.savefig(OUT / 'conformal_per_battery_heatmap.png', dpi=150); plt.show()

## 2. Ablation table — contribution of each component

Train four short models to isolate the effect of:

- **numeric-only** — drop the semantic modality entirely (set embeddings to zero).
- **multimodal-MSE** — baseline Phase-3 setup.
- **multimodal + PINN** — adds physics-informed loss.

Each trained for a small number of epochs so the table is reproducible within a few minutes on CPU. Epoch count is deliberately modest; the numbers are meaningful only *relative* to each other within this table.

In [ ]:
from phase4.pinn_loss import PINNCompositeLoss, PINNWeights
from torch.utils.data import Dataset, DataLoader

class Ds(Dataset):
    def __init__(self, frame, seq, em, y_):
        self.frame = frame.reset_index(drop=True); self.seq = seq; self.em = em; self.y = y_
        self.bat_codes = pd.Categorical(self.frame['battery_id']).codes.astype(np.int64)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        return {'seq': torch.from_numpy(self.seq[i]), 'em': torch.from_numpy(self.em[i]),
                'y': torch.tensor(self.y[i], dtype=torch.float32),
                'cyc': torch.tensor(r['cycle_number'], dtype=torch.float32),
                'T':   torch.tensor(r.get('temp_mean', 0.0), dtype=torch.float32),
                'cap': torch.tensor(r.get('capacity', 0.0), dtype=torch.float32),
                'r':   torch.tensor(r.get('dcir', 0.0), dtype=torch.float32),
                'bat': torch.tensor(self.bat_codes[i], dtype=torch.long)}

def loader(frame, seq, em, y_, bs=64, shuffle=True):
    return DataLoader(Ds(frame, seq, em, y_), batch_size=bs, shuffle=shuffle)

def fresh():
    return CrossAttentionSOHModel(
        numeric_input_dim=len(feature_cols), semantic_input_dim=emb['train'].shape[1],
        hidden_dim=cfg['hidden_dim'], n_attention_heads=cfg['n_attention_heads'],
        n_transformer_layers=cfg['n_transformer_layers']).to(device)

@torch.no_grad()
def evaluate_on(model, seq_np, emb_np, y_true):
    model.eval(); out = []
    for i in range(0, len(seq_np), 128):
        out.append(model(torch.from_numpy(seq_np[i:i+128]).to(device),
                         torch.from_numpy(emb_np[i:i+128]).to(device))['soh'].cpu().numpy())
    p = np.concatenate(out)
    return {'MAE': float(np.mean(np.abs(p-y_true))),
            'RMSE': float(np.sqrt(np.mean((p-y_true)**2))),
            'preds': p}

def mono_rate(preds, frame):
    bad = total = 0
    for _, g in frame.assign(_p=preds).groupby('battery_id'):
        gg = g.sort_values('cycle_number')['_p'].values
        if len(gg) < 2: continue
        bad += int((gg[1:] - gg[:-1] > 0).sum()); total += len(gg) - 1
    return bad / max(total, 1)

In [ ]:
def train_run(variant, epochs=15):
    torch.manual_seed(7)
    model = fresh(); opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    em_train = emb['train'] if variant != 'numeric_only' else np.zeros_like(emb['train'])
    em_val = emb['val']     if variant != 'numeric_only' else np.zeros_like(emb['val'])
    em_test = emb['test']   if variant != 'numeric_only' else np.zeros_like(emb['test'])
    train_dl = loader(splits['train'], X_seq['train'], em_train, y['train'])
    if variant == 'pinn':
        crit = PINNCompositeLoss(weights=PINNWeights(monotonicity=0.05, arrhenius=0.02, capacity_resistance=0.05))
    else:
        crit = torch.nn.MSELoss()
    for ep in range(epochs):
        model.train()
        for b in train_dl:
            opt.zero_grad()
            out = model(b['seq'].to(device), b['em'].to(device))['soh']
            if variant == 'pinn':
                loss = crit(out, b['y'].to(device), b['cyc'].to(device), b['T'].to(device),
                            b['cap'].to(device), b['r'].to(device), b['bat'].to(device))
            else:
                loss = crit(out, b['y'].to(device))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    res = evaluate_on(model, X_seq['test'], em_test, y['test'])
    res['mono_viol'] = mono_rate(res['preds'], splits['test'])
    return model, res

print('training numeric_only...');    _, r_num   = train_run('numeric_only', epochs=15)
print('training multimodal-MSE...');  _, r_mmse  = train_run('mse',           epochs=15)
print('training multimodal+PINN...'); _, r_pinn  = train_run('pinn',          epochs=15)

In [ ]:
ablation = pd.DataFrame([
    {'model': 'Numeric-only (no semantic modality)',  **{k:r_num[k]  for k in ('MAE','RMSE','mono_viol')}},
    {'model': 'Multimodal (MSE only)',                **{k:r_mmse[k] for k in ('MAE','RMSE','mono_viol')}},
    {'model': 'Multimodal + PINN composite loss',     **{k:r_pinn[k] for k in ('MAE','RMSE','mono_viol')}},
])
ablation['MAE'] = ablation['MAE'].round(3); ablation['RMSE'] = ablation['RMSE'].round(3); ablation['mono_viol'] = ablation['mono_viol'].round(3)
ablation.to_csv(OUT / 'ablation.csv', index=False)
ablation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
xs = np.arange(len(ablation)); wx = 0.38
ax.bar(xs - wx/2, ablation['MAE'],  width=wx, label='MAE')
ax.bar(xs + wx/2, ablation['RMSE'], width=wx, label='RMSE', alpha=0.85)
ax.set_xticks(xs); ax.set_xticklabels(['numeric-only','MSE','MSE+PINN'], rotation=0)
ax.set_ylabel('SOH error'); ax.legend(); ax.grid(alpha=0.3, axis='y')
ax.set_title('Point-error ablation')
ax = axes[1]
ax.bar(xs, ablation['mono_viol']*100, color=['#aaa','#5b8def','#7c3aed'])
ax.set_xticks(xs); ax.set_xticklabels(['numeric-only','MSE','MSE+PINN'])
ax.set_ylabel('Monotonicity violation rate (%)'); ax.grid(alpha=0.3, axis='y')
ax.set_title('Physics-consistency ablation (lower is better)')
plt.tight_layout(); plt.savefig(OUT / 'ablation.png', dpi=150); plt.show()

## 3. Weibull RUL calibration — PIT histogram

For a well-calibrated survival model, the probability-integral-transform of observed RUL under the predicted Weibull CDF should be uniform on [0, 1]. Bar heights close to 1/n_bins across all bins = well-calibrated.

In [ ]:
from phase4.survival_head import CrossAttentionWithRULHead, weibull_nll, compute_rul_targets

rul_df = compute_rul_targets(df, eol_threshold=80.0)
rul_lookup = rul_df.set_index(['battery_id','cycle_number'])[['rul','event']]
rul_sp = {k: splits[k].assign(rul=rul_lookup.loc[list(zip(splits[k]['battery_id'], splits[k]['cycle_number'])), 'rul'].values,
                              event=rul_lookup.loc[list(zip(splits[k]['battery_id'], splits[k]['cycle_number'])), 'event'].values)
          for k in splits}

rul_model = CrossAttentionWithRULHead(base_model).to(device)
rul_model.freeze_base()
opt = torch.optim.Adam(rul_model.rul_head.parameters(), lr=5e-3)

def rul_batches(k, bs=64, shuffle=True):
    s = rul_sp[k]
    items = list(zip(X_seq[k], emb[k], s['rul'].values.astype(np.float32), s['event'].values.astype(np.int64)))
    def coll(b):
        seq = torch.tensor(np.stack([x[0] for x in b])); em = torch.tensor(np.stack([x[1] for x in b]))
        ru = torch.tensor(np.array([x[2] for x in b])); ev = torch.tensor(np.array([x[3] for x in b]))
        return seq, em, ru, ev
    return DataLoader(items, batch_size=bs, shuffle=shuffle, collate_fn=coll)

for ep in range(25):
    rul_model.train()
    for seq, em, ru, ev in rul_batches('train'):
        seq, em, ru, ev = seq.to(device), em.to(device), ru.to(device), ev.to(device)
        opt.zero_grad()
        out = rul_model(seq, em)
        loss = weibull_nll(out['rul_scale'], out['rul_shape'], ru, ev)
        if not torch.isfinite(loss): continue
        loss.backward(); torch.nn.utils.clip_grad_norm_(rul_model.rul_head.parameters(), 1.0); opt.step()
print('RUL head trained.')

In [ ]:
# Collect predictions across cal+val+test where event==1 (uncensored only); compute PIT.
rul_model.eval()
pit_values = []
with torch.no_grad():
    for split_name in ('cal','val','test'):
        for seq, em, ru, ev in rul_batches(split_name, shuffle=False):
            seq, em, ru, ev = seq.to(device), em.to(device), ru.to(device), ev.to(device)
            out = rul_model(seq, em)
            scale = out['rul_scale']; shape = out['rul_shape']
            cdf = 1.0 - torch.exp(-torch.pow((ru.clamp(min=1e-6)/scale.clamp(min=1e-6)), shape))
            mask = (ev == 1) & (ru > 0)
            if mask.any():
                pit_values.append(cdf[mask].cpu().numpy())
if pit_values:
    pit = np.concatenate(pit_values)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(pit, bins=10, range=(0,1), density=True, edgecolor='black', alpha=0.75)
    ax.axhline(1.0, color='red', linestyle='--', label='uniform (perfect calibration)')
    ax.set_xlabel('PIT value = Weibull CDF(observed RUL)')
    ax.set_ylabel('Density')
    ax.set_title(f'PIT histogram (n={len(pit)} uncensored)')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT / 'weibull_pit.png', dpi=150); plt.show()
    print(f'PIT mean={pit.mean():.3f}, std={pit.std():.3f}')
else:
    print('No uncensored samples \u2014 PIT undefined for this split.')

## 4. RAP cost-benefit — does richer text improve downstream regression?

Replace the template semantic embedding with the RAP-generated text (embedded with SBERT), and re-score the base cross-attention model on a sample of test cycles.

In [ ]:
from phase4.rap import build_rap, LLMConfig
from feature_to_text import FeatureToText, convert_features_to_text
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('all-mpnet-base-v2')

fall_conv = FeatureToText()
fall_conv.set_reference_values(splits['train'].iloc[0].to_dict())
def template_fallback(feats):
    return fall_conv.convert_single(feats).full_description

rap = build_rap(splits['train'].reset_index(drop=True), feature_cols, train_texts, k=5,
                cache_dir='data/phase4/rap_cache',
                llm_config=LLMConfig(model=os.environ.get('OPENAI_MODEL','gpt-4o-mini')),
                template_fallback=template_fallback)

RAP_N = min(80, len(splits['test']))
idx = np.random.RandomState(0).choice(len(splits['test']), size=RAP_N, replace=False)
sub = splits['test'].iloc[idx].reset_index(drop=True)
rap_texts = [rap.diagnose(r.to_dict()).text for _, r in sub.iterrows()]
tpl_texts = convert_features_to_text(sub)['full_description'].tolist()

rap_emb = sbert.encode(rap_texts, show_progress_bar=False).astype(np.float32)
tpl_emb = sbert.encode(tpl_texts, show_progress_bar=False).astype(np.float32)
sub_seq = X_seq['test'][idx]; sub_y = y['test'][idx]

def eval_pair(seqs, embs, ys):
    p = predict_soh_batch(seqs, embs)
    return {'MAE': float(np.mean(np.abs(p-ys))), 'RMSE': float(np.sqrt(np.mean((p-ys)**2)))}

rap_metrics = eval_pair(sub_seq, rap_emb, sub_y)
tpl_metrics = eval_pair(sub_seq, tpl_emb, sub_y)
rap_df = pd.DataFrame([{'variant':'template', **tpl_metrics}, {'variant':'RAP (retrieval+LLM)', **rap_metrics}])
rap_df.to_csv(OUT / 'rap_vs_template.csv', index=False)
rap_df

## 5. Summary

All artefacts written to `results/phase4_evaluation/`:
- `conformal_coverage.csv`, `conformal_coverage_per_battery.csv`, `conformal_calibration.png`, `conformal_per_battery_heatmap.png`
- `ablation.csv`, `ablation.png`
- `weibull_pit.png`
- `rap_vs_template.csv`

These are drop-in thesis figures/tables.

In [ ]:
print('Artefacts produced:')
for f in sorted(OUT.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')